In [1]:
from os import environ

from dotenv import load_dotenv
import influxdb_client
from influxdb_client import InfluxDBClient, Point, WritePrecision
from influxdb_client.client.write_api import SYNCHRONOUS
import pandas as pd

## Shift the data to the current year

In [2]:
historical_data = pd.read_csv("./imputed_data.csv", index_col=0, parse_dates=True)
historical_data

,power
time,
2022-01-01 00:00:00+00:00,235.753968
2022-01-01 00:15:00+00:00,207.167347
2022-01-01 00:30:00+00:00,222.972332
2022-01-01 00:45:00+00:00,235.083665
2022-01-01 01:00:00+00:00,203.906883
...,...
2022-12-31 22:45:00+00:00,2456.169435
2022-12-31 23:00:00+00:00,2405.723333
2022-12-31 23:15:00+00:00,2371.020000


In [3]:
offset = pd.Timedelta(3*365 + 1, "d")  # 3 years and one leap day in 2024
shifted_data = historical_data.set_index(historical_data.index + offset)
shifted_data

,power
time,
2025-01-01 00:00:00+00:00,235.753968
2025-01-01 00:15:00+00:00,207.167347
2025-01-01 00:30:00+00:00,222.972332
2025-01-01 00:45:00+00:00,235.083665
2025-01-01 01:00:00+00:00,203.906883
...,...
2025-12-31 22:45:00+00:00,2456.169435
2025-12-31 23:00:00+00:00,2405.723333
2025-12-31 23:15:00+00:00,2371.020000


## Add tags to match our format

In [4]:
shifted_data["domain"] = "sensor"
shifted_data["entity_id"] = "smart_meter_replay"
shifted_data

,power,domain,entity_id
time,,,
2025-01-01 00:00:00+00:00,235.753968,sensor,smart_meter_replay
2025-01-01 00:15:00+00:00,207.167347,sensor,smart_meter_replay
2025-01-01 00:30:00+00:00,222.972332,sensor,smart_meter_replay
2025-01-01 00:45:00+00:00,235.083665,sensor,smart_meter_replay
2025-01-01 01:00:00+00:00,203.906883,sensor,smart_meter_replay
...,...,...,...
2025-12-31 22:45:00+00:00,2456.169435,sensor,smart_meter_replay
2025-12-31 23:00:00+00:00,2405.723333,sensor,smart_meter_replay
2025-12-31 23:15:00+00:00,2371.020000,sensor,smart_meter_replay


## Connecting to InfluxDB
In our InfluxDB instance, create an access token and pit it into a .env file.

In [5]:
load_dotenv()
token = environ["INFLUXDB_TOKEN"]

org = "homeassist"
# url = "http://asr-demo6-usg.sb.dfki.de"
url = "http://localhost:8086"
bucket="homeassist"

## Writing to the data base
According to the doc here: https://influxdb-client.readthedocs.io/en/stable/api.html#influxdb_client.WriteApi.write

In [6]:
with influxdb_client.InfluxDBClient(url=url, token=token, org=org) as client:
    write_api = client.write_api(write_options=SYNCHRONOUS)
       
    write_api.write(
        bucket=bucket,
        org="homeassist",
        record=shifted_data,
        data_frame_measurement_name="W",
        data_frame_tag_columns=["domain", "entity_id"],
)